In [29]:
import random
from collections import Counter
from typing import List, Tuple
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, Dataset

In [30]:
SEED = 42
EMBED_DIM = 100
WINDOW_SIZE = 2
NEGATIVE_SAMPLES = 5
BATCH_SIZE = 512
EPOCHS = 3
LR = 0.002
NUM_DOCUMENTS = 100
MAX_VOCAB = 5000
# Classification
N_CLASSES = 2
MAX_LEN = 120
CLS_BATCH_SIZE = 128
CLS_EPOCHS = 3
CLS_LR = 3e-3
ATTN_HIDDEN = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PAD, UNK = 0, 1

In [31]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

In [32]:
def tokenize(x: str) -> list[str]:
    return x.lower().split()

def load_imdb_texts():
    imdb_in_dataset = load_dataset("stanfordnlp/imdb")

    train_texts = imdb_in_dataset["train"]["text"]
    train_labels = imdb_in_dataset["train"]["label"] # 0=neg, 1=pos

    test_texts = imdb_in_dataset["test"]["text"]
    test_labels = imdb_in_dataset["test"]["label"]

    print(f"#train: {len(train_texts)}, #test: {len(test_texts)}")
    print(f"train_texts[0][:2] = {train_texts[0][:30]}")
    print(f"train_labels[0:2] = {train_labels[0:20]}")

    return train_texts, train_labels, test_texts, test_labels


In [33]:
def build_vocab(texts: List[str], max_vocab: int) -> Tuple[dict, dict]:
    cnt = Counter()
    for t in texts:
        cnt.update(tokenize(t))
    vocab = ["<pad>", "<unk>"] + [w for w, _ in cnt.most_common(max_vocab - 2)]
    word2idx = {w:i for i,w in enumerate(vocab)}
    idx2word = {i:w for w,i in word2idx.items()}
    return word2idx, idx2word

    # 단어를 카운트함
    # cnt["the"] = 120345
    # cnt["movie"] = 65421
    # cnt["great"] = 3421


In [34]:
def texts_to_indexed_docs(texts: List[str], word2idx: dict) -> List[List[int]]:
    return [[word2idx.get(tok, UNK) for tok in tokenize(t)] for t in texts]

In [35]:
def make_unigram_probs(indexed_docs: List[List[int]], vocab_size: int, power: float = 0.75) -> np.ndarray:
    freqs = np.zeros(vocab_size, dtype=np.float64)
    for doc in indexed_docs:
        for i in doc:
           freqs[i] += 1
    freqs[PAD] = 0.0
    freqs[UNK] = 0.0
    probs = freqs ** power
    probs /= probs.sum()
    return probs

In [36]:
def encode_fixed(text: str, word2idx: dict, max_len: int) -> List[int]:
    ids = [word2idx.get(tok, UNK) for tok in tokenize(text)[:max_len]]
    if len(ids) < max_len:
        ids += [PAD] * (max_len - len(ids))
    return ids

In [37]:
class BaseEmbed(nn.Module):
    def __init__(self, vocab_size, emb_dim, init_weight: np.ndarray):
        super().__init__()
        if init_weight is not None:
          self.emb = nn.Embedding.from_pretrained(torch.tensor(init_weight), padding_idx=
          PAD, freeze=False)
        else:
          self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)

In [38]:
class MeanPoolClassifier(BaseEmbed):
    def __init__(self, vocab_size, emb_dim, n_classes, init_weight):
        super().__init__(vocab_size, emb_dim, init_weight)
        self.fc = nn.Linear(emb_dim, n_classes)
    def forward(self, x):
        e = self.emb(x) # [B,T,E]
        mean = e.mean(dim=1) # [B,E]
        return self.fc(mean)

# **Skip_gram**

In [39]:
def build_skipgram_pairs_docs(indexed_docs: list[list[int]], window_size:int =2) -> list[Tuple[int, int]]:
  output = []
  print("idx in doc :", indexed_docs[0])
  for doc in  indexed_docs:
    for c_i , c in enumerate(doc):
      left = max(0,c_i - window_size)
      right = min(c_i + window_size, len(doc) - 1)
      for j in range(left, right + 1):
        if j != c_i:
          output.append((c, doc[j]))
  return output


In [40]:
class SkipGramDataset(Dataset):
    def __init__(self, pairs: List[Tuple[int, int]], probs: np.ndarray, vocab_size: int, k: int):
      self.pairs= pairs
      self.probs= probs
      self.vocab_size= vocab_size
      self.k= k

    def __len__(self):
      return len(self.pairs)

    def __getitem__(self, idx):

        center, pos= self.pairs[idx]

        negs= np.random.choice(self.vocab_size, size=self.k, replace=True, p=self.probs)

        return(
        torch.tensor(center),
        torch.tensor(pos),
        torch.tensor(negs),
        )

In [41]:
class SkipGramNS(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.in_embed = nn.Embedding(vocab_size, embed_dim)
        self.out_embed = nn.Embedding(vocab_size, embed_dim)
        nn.init.uniform_(self.in_embed.weight, a=-0.5/embed_dim, b=0.5/embed_dim)
        nn.init.uniform_(self.out_embed.weight, a=-0.5/embed_dim, b=0.5/embed_dim)

    def forward(self, cent, pos, neg):
      # B: mini-batch size, K: #(negative samples), D: self.embed_dim
      # cent: (B), pos: (B), neg: (B, K)
      cent= self.in_embed(cent) # (B, D)
      pos= self.in_embed(pos) # (B, D)
      neg= self.in_embed(neg) # (B, K, D)
      pos_mult= (cent* pos) # (B, D)
      pos_inner= pos_mult.sum(dim=-1) # (B)
      cent_2= cent.reshape(-1, 1, self.embed_dim) # (B, 1, D)
      neg_mult= cent_2* neg# (B, K, D)
      neg_inner= neg_mult.sum(dim=-1) # (B, K)
      pos_loss= -nn.functional.logsigmoid(pos_inner) # (B)
      neg_loss= -nn.functional.logsigmoid(-neg_inner).sum(dim=-1) # (B)
      return (pos_loss+neg_loss).mean()

    def get_input_embeddings(self):
      return self.in_embed.weight.detach().cpu().numpy()


In [42]:
def train_1_epoch(model: nn.Module, loader: DataLoader, opt: torch.optim.Optimizer, device, loss_f):
    model.train()
    total_loss = 0.0
    n = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        batch_size = X.shape[0]
        opt.zero_grad()
        out = model(X)
        loss = loss_f(out, y)
        loss.backward()
        opt.step()
        total_loss += loss.item() * batch_size
        n += batch_size
    return total_loss / n

In [43]:
@torch.no_grad()
def eval_model(model, loader, device):
  model.eval()
  ys, preds = [], []
  for X, y in loader:
      X, y = X.to(device), y.to(device)
      p = model(X).argmax(dim=1)
      ys += y.tolist()
      preds += p.tolist()
  acc = accuracy_score(ys, preds)
  return acc

# **main**

In [ ]:
def main():
    train_texts, train_labels, test_texts, test_labels = load_imdb_texts()
    word2idx, idx2word = build_vocab(train_texts, MAX_VOCAB)
    vocab_size = len(word2idx)

    sgns_texts = train_texts[:NUM_DOCUMENTS]
    indexed_docs = texts_to_indexed_docs(sgns_texts, word2idx)
    pairs = build_skipgram_pairs_docs(indexed_docs, WINDOW_SIZE)

    probs = make_unigram_probs(indexed_docs, vocab_size)
    dataset = SkipGramDataset(pairs, probs, vocab_size, NEGATIVE_SAMPLES)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    model = SkipGramNS(vocab_size, EMBED_DIM).to(DEVICE)
    optim = torch.optim.Adam(model.parameters(), lr=LR)


    model.train()
    for epoch in range(1, EPOCHS + 1):
        epoch_loss = 0.0
        for center, pos, neg in loader:
            center, pos, neg = center.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE)
            loss = model(center, pos, neg)
            loss.backward()
            optim.step()
            optim.zero_grad()
            epoch_loss += loss.item()
        print(f"[Epoch {epoch}] mean loss: {epoch_loss / len(loader):.4f}")
    embeds = model.get_input_embeddings()

    tr_texts_enc = [encode_fixed(t, word2idx, MAX_LEN) for t in train_texts]
    te_texts_enc = [encode_fixed(t, word2idx, MAX_LEN) for t in test_texts]
    X_tr = torch.tensor(tr_texts_enc)
    y_tr = torch.tensor(train_labels)
    X_te = torch.tensor(te_texts_enc)
    y_te = torch.tensor(test_labels)
    tr_ds = torch.utils.data.TensorDataset(X_tr, y_tr)
    te_ds = torch.utils.data.TensorDataset(X_te, y_te)
    tr_loader = DataLoader(tr_ds, batch_size=CLS_BATCH_SIZE, shuffle=True, drop_last=True)
    te_loader = DataLoader(te_ds, batch_size=CLS_BATCH_SIZE)
    # ...
    # ...
    set_seed(SEED)
    cls_model = MeanPoolClassifier(vocab_size, EMBED_DIM, N_CLASSES, embeds)
    cls_model = cls_model.to(DEVICE)
    opt = torch.optim.Adam(cls_model.parameters(), lr=CLS_LR)
    loss_f = nn.CrossEntropyLoss()
    for ep in range(1, CLS_EPOCHS + 1):
      tr_loss = train_1_epoch(cls_model, tr_loader, opt, DEVICE, loss_f)
      print(f" Ep {ep:02d} | loss {tr_loss:.4f}")
      te_acc = eval_model(cls_model, te_loader, DEVICE)
      print(f" => Test Accuracy: {te_acc:.4f}")
if __name__ == "__main__":
    main()

#train: 25000, #test: 25000
train_texts[0][:2] = I rented I AM CURIOUS-YELLOW f
train_labels[0:2] = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
idx in doc : [9, 1486, 9, 226, 1, 34, 57, 433, 1495, 77, 5, 35, 2, 1, 11, 3471, 12, 50, 12, 14, 82, 702, 8, 1, 9, 81, 510, 11, 29, 82, 12, 14, 1, 31, 2443, 1, 46, 12, 125, 737, 6, 2764, 10, 4218, 1935, 99, 3, 376, 5, 129, 1127, 1, 9, 61, 62, 6, 67, 10, 16, 1, 13, 93, 131, 7, 1, 197, 3, 185, 4380, 659, 1669, 699, 1, 36, 438, 6, 781, 292, 53, 64, 43, 506, 8, 928, 53, 438, 6, 1182, 42, 1, 6, 242, 45, 391, 5, 797, 19, 48, 2, 952, 1, 199, 43, 721, 954, 1600, 130, 15, 2, 3472, 408, 4, 1977, 1600, 8, 2, 2351, 1, 8, 187, 2170, 1, 4, 2268, 1, 5, 1, 43, 59, 1, 19, 1, 53, 41, 454, 17, 42, 659, 1, 1, 4, 1114, 1, 13, 1063, 1068, 87, 43, 9, 226, 1, 7, 11, 1941, 181, 1861, 10, 14, 1127, 1, 1593, 2, 454, 4, 1375, 159, 22, 156, 4, 230, 1, 55, 96, 44, 23, 339, 37, 45, 1, 95, 1, 133, 57, 1, 432, 153, 12, 1, 8, 848, 454, 4, 1375, 22, 3, 591, 1, 8,

In [ ]:
class ATT(nn.module):
  def __init__(self):
    super().__init__()
    self.lin1 = nn.Linear(in_dim, 100)
    self.relu1 = nn.ReLU()
    self.lin2 = nn.Linear(100, 50)

  def forward(self, v, mask): #v: (B ,L, D) #s:(B, L, 1)
    s = self.lin2(self.relu1(self.lin1(v)))
    s.masked_fill_(mask, -float("inf"))
    alpha = torch.softmax(s, dim=1)
    alpha_v = alpha * v # (B, L, D)
    return alpha_v.sum(dim=1)#(B,D)

class AttpoolClassifier(BaseEmbed):
  def __init__(self, vocab_size, emb_dim, n_classes, init_weight=None):
    super().__init__(vocab_size, emb_dim, init_weight)
    self.att = ATT(emb_dim)
    self.lin2 = nn.linear(100,50)
    self.relu2 = nn.ReRU()
    self.lin3 = nn.linear(50, n_classes)

  def forward(self, x):
    e = self.emb(x) #(B, L, D)
    v = self.att(e, x==PAD) #(B, D)
    o = self.relu1(self.lin1(v))
    o = self.relu2(self.lin2(o))

# **self_attention**

In [ ]:
class SimpleSelfAtt(nn.Module):
  def __init__(self,in_dim):
    super().__init__()
    self.lin_q = nn.linear(in_dim, in_dim, bias=False)
    self.lin_k = nn.linear(in_dim, in_dim, bias=False)
    self.lin_v = nn.linear(in_dim, in_dim, bias=False)

  def forward(self,x:torch.Tensor,pad_mask:torch.Tensor):
    #pad_mask : (B,T)
    B, T, _ = x.shape
    q = self.lin_q(x) #(B,T,D)
    k = self.lin_k(x)
    v = self.lin_v(x)
    k_ = k.transpose(1,2) #(B,D,T)
    scores = torch.bmm(q,k_.transpose((1,2)) / self.emb_dim ** 0.5) #(B,T,T)
    scores_mask = scores.masked_fill(pad_mask.unsqueeze(1), -float("inf")) #텍스트마다 길이가 다르므로 mask 처리
    alpha = torch.softmax(scores, dim = -1) #루트 D를 나누는 이유 공부해야됨 (B,T,T)
    output = torch.bmm(alpha ,v) #(B, T, D)
    #단어 벡터를 새롭게 변환
    return output

# **Transformer**

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self, emb_dim:int , ff_hidden:int =32 ):
      super().__init__()
      self.ln1 = nn.LayerNorm(emb_dim)
      self.ln2 = nn.LayerNorm(emb_dim)
      self.att = SimpleSelfAtt(emb_dim)
      self.lin1 = nn.linear(emb_dim, ff_hidden)
      self.relu = nn.ReLU()
      self.lin2 = nn.linear(ff_hidden, emb_dim)

  def forward(self, x, pad_mask=None):
      x = x + self.att((x,pad_mask))
      x = self.ln1(x)
      x = x + self.lin(self.relu(self.lin1(x)))
      x = self.ln2(x)

In [ ]:
class MultiHeadSelfAttention2(nn.Module):
  def __init__(self,emb_dim , num_heads =4):
    super().__init__()


    self.head_dim = in_dim // self.head

    self.lin_q = nn.linear(in_dim, in_dim, bias=False)
    self.lin_k = nn.linear(in_dim, in_dim, bias=False)
    self.lin_v = nn.linear(in_dim, in_dim, bias=False)

  def forward(self,x:torch.Tensor,pad_mask:torch.Tensor):
    #pad_mask : (B,T)
    B, T, D = x.shape
    h = self.num_heads
    Dh = self.head_dim
    q = self.lin_q(x) #(B,T,D)
    k = self.lin_k(x)
    v = self.lin_v(x)
    q = q.reshape(B,T,h,Dh).transpose(1,2)
    k = k.reshape(B,T,h,Dh).transpose(1,2)
    V = v.reshape(B,T,h,Dh).transpose(1,2)
    Q_ = q #(B,H,T,Dh)
    K_ = k.transpose(-2, -1) #(B,H,T,Dh) -> (B, H, Dh, T)
    scores = torch.matmul(Q_,K_)
    if pad_mask is not None:
      # pad_mask : [B,T] -> [B,1,T]
      # old_scores : [B,T,T]
      pad_mask = pad_mask.reshape(B, 1, 1, T)
      scores = scores.masked_fill(pad_mask, -1e9)
    alpha = torch.softmax(scores, dim= -1) #[B, H, T, T]
    V_ = V #[B, H, T, Dh]
    out = torch.mat_mul(alpha, V_) #[B, H, T, Dh]
    out2 =out.transpose(1,2).reshape(B,T,E) #[B,T,E]
    return out2